# The Vanished Clause

Rebuild a CFR section's wording *before* an amendment, given how it reads *after*, the
wording the amendment removed (as loose fragments) and a set of `[GAP]` marks — about
one in four of which is a place a fragment really came out of.

Three progressive submissions:

1. **Marks removed, nothing put back** — the honest floor. Scores 0 by construction.
2. **Bigram-count heuristic** — no training labels. Each fragment goes where the word
   before the mark is most often followed by the fragment's first word.
3. **Learned pair scorer + one-to-one assignment** — a linear model over
   (mark, fragment) pairs trained on `train_slots.csv`, then the best matching of
   fragments to marks per section.

Everything is learned from the training files. No pretrained model, no external text,
and every test row is predicted on its own: n-gram counts come from the training text
and from the row's own shown text, never from `test.csv` as a corpus.

In [1]:
import os, re, json, math, warnings
warnings.filterwarnings("ignore")     # lbfgs stops at max_iter; the fit is deterministic
from collections import Counter, defaultdict
import numpy as np
import pandas as pd

SEED = 20260930
np.random.seed(SEED)
DATA = next(p for p in ("/data", "dataset/public",
                        "/kaggle/input/ecfr-amendment-reversal/public")
            if os.path.exists(os.path.join(p, "test.csv")))
OUT = "./working" if os.path.isdir("./working") else "."
os.makedirs(OUT, exist_ok=True)

train = pd.read_csv(os.path.join(DATA, "train.csv"))
test  = pd.read_csv(os.path.join(DATA, "test.csv"))
slots = pd.read_csv(os.path.join(DATA, "train_slots.csv"))
print("train", train.shape, "| test", test.shape, "| train_slots", slots.shape)

train (410, 4) | test (353, 3) | train_slots (4631, 3)


## Reading a row

`text_after` splits on `[GAP]` into pieces of text; mark `k` sits between piece `k` and
piece `k+1`. `removed_fragments` is a JSON list. A prediction is the pieces joined back
together with fragments dropped into the chosen marks.

In [2]:
MARK = "[GAP]"
TOKEN = re.compile(r"\w+|[^\w\s]")
tok = TOKEN.findall

def parse(text_after):
    """(tokens of the clean text, token offset of every mark, text pieces)."""
    pieces = text_after.split(MARK)
    toks, offs = [], []
    for k, piece in enumerate(pieces):
        toks += tok(piece)
        if k < len(pieces) - 1:
            offs.append(len(toks))
    return toks, offs, pieces

def render(pieces, put):
    """Rebuild the text with put[mark] = [fragment, ...]; every mark removed."""
    out = []
    for k, piece in enumerate(pieces):
        out.append(piece)
        if k < len(pieces) - 1:
            out.append(" " + " ".join(put.get(k, [])) + " ")
    return re.sub(r"\s+", " ", "".join(out)).strip()

def rows(df):
    out = []
    for r in df.itertuples(index=False):
        toks, offs, pieces = parse(r.text_after)
        out.append({"row_id": r.row_id, "toks": toks, "slots": offs, "pieces": pieces,
                    "frags": json.loads(r.removed_fragments),
                    "before": getattr(r, "text_before", None)})
    return out

TR, TE = rows(train), rows(test)
labels = defaultdict(dict)                 # row_id -> {fragment_index: gap_index}
for r in slots.itertuples(index=False):
    labels[r.row_id][r.fragment_index] = r.gap_index

n_frag = [len(r["frags"]) for r in TR]; n_mark = [len(r["slots"]) for r in TR]
print("median fragments / marks per train row:", int(np.median(n_frag)), "/", int(np.median(n_mark)))
print("share of marks that are real:", round(sum(n_frag) / sum(n_mark), 3))
ex = TR[76]
print("\nexample", ex["row_id"], "| fragments:", ex["frags"], "| true (mark <- fragment):",
      sorted((g, f) for f, g in labels[ex["row_id"]].items()))

median fragments / marks per train row: 4 / 16
share of marks that are real: 0.248

example tr_00076 | fragments: [',', '§ 250.1603 (SOP only),'] | true (mark <- fragment): [(5, 1), (7, 0)]


## The metric, reimplemented

Worth having locally: it is the only honest way to compare approaches. The grader sees
only the submitted text. It aligns the shown text to the prediction, takes the tokens
the prediction added, keys each by the gap it sits in, and compares them with what the
true prior wording adds.

Validation uses the last fifth of `train.csv`. The file is in source order, so that
tail is a different set of CFR parts from the head — the same kind of split as the real
test set.

In [3]:
import difflib
ANCHOR = 8

def _lcs_gaps(s, o):
    n, m = len(s), len(o)
    if m == 0: return []
    if n == 0: return [(0, t) for t in o]
    prev = [0]*(m+1); choices = []
    for i in range(1, n+1):
        cur = [0]*(m+1); row = bytearray(m+1); si = s[i-1]
        for j in range(1, m+1):
            if si == o[j-1]: cur[j] = prev[j-1]+1; row[j] = 1
            elif prev[j] >= cur[j-1]: cur[j] = prev[j]; row[j] = 2
            else: cur[j] = cur[j-1]; row[j] = 3
        choices.append(row); prev = cur
    out = []; i, j = n, m
    while i > 0 and j > 0:
        c = choices[i-1][j]
        if c == 1: i -= 1; j -= 1
        elif c == 2: i -= 1
        else: out.append((i, o[j-1])); j -= 1
    while j > 0: out.append((0, o[j-1])); j -= 1
    return out

def changed_in_gaps(s, o):
    sm = difflib.SequenceMatcher(None, s, o, autojunk=False)
    out, si, oi = [], 0, 0
    for a0, b0, size in sm.get_matching_blocks():
        if size < ANCHOR and size != 0: continue
        out += [(g + si, t) for g, t in _lcs_gaps(s[si:a0], o[oi:b0])]
        si, oi = a0 + size, b0 + size
    return out

def score_row(pred, shown_toks, truth):
    changed = Counter(changed_in_gaps(shown_toks, tok(truth)))
    proposed = Counter(changed_in_gaps(shown_toks, tok(pred.replace(MARK, " "))))
    hits = sum(min(c, proposed.get(k, 0)) for k, c in changed.items())
    if hits == 0: return 0.0
    p, r = hits / sum(proposed.values()), hits / sum(changed.values())
    return 2 * p * r / (p + r)

n_fit = int(len(TR) * 0.8)
FIT, VAL = TR[:n_fit], TR[n_fit:]
def evaluate(fn, rowset=VAL):
    return float(np.mean([score_row(fn(r), r["toks"], r["before"]) for r in rowset]))
print(len(FIT), "fit rows |", len(VAL), "validation rows")
print("true prior wording (val):", round(evaluate(lambda r: r["before"]), 4))

328 fit rows | 82 validation rows


true prior wording (val): 1.0


## Submission 1 — marks removed, nothing put back

The sample submission. It is ~98% of the characters and worth 0 of the score: the
metric only counts wording that was put back.

In [4]:
def stage1(row):
    return render(row["pieces"], {})

print("marks removed (val):", round(evaluate(stage1), 4))
pd.DataFrame({"row_id": [r["row_id"] for r in TE], "text_before": [stage1(r) for r in TE]}) \
  .to_csv(os.path.join(OUT, "submission.csv"), index=False)
print("wrote submission.csv")

marks removed (val): 0.0
wrote submission.csv


## Submission 2 — bigram-count heuristic (no labels)

Count token n-grams over the training text — both versions of every training section —
and, separately, over the row's own shown text. A fragment fits a mark if the word before the
mark is often followed by the fragment's first word, and the fragment's last word by
what comes next. The amendment may have put *new* words right after a real mark, so
"what comes next" looks a few tokens ahead too.

Fragments are then matched to marks one-to-one (each mark lost at most one fragment)
with the Hungarian algorithm.

In [5]:
from scipy.optimize import linear_sum_assignment

class Counts:
    def __init__(self, texts=()):
        self.uni, self.big, self.tri = Counter(), Counter(), Counter()
        for t in texts: self.add(t)
    def add(self, t):
        self.uni.update(t); self.big.update(zip(t, t[1:])); self.tri.update(zip(t, t[1:], t[2:]))

class _Diff:
    def __init__(self, a, b): self.a, self.b = a, b
    def get(self, k, d=0): return self.a.get(k, 0) - self.b.get(k, 0)

class Minus:
    """`total` with one fold's texts taken out, read-only."""
    def __init__(self, total, part):
        self.uni, self.big, self.tri = (_Diff(total.uni, part.uni), _Diff(total.big, part.big),
                                        _Diff(total.tri, part.tri))

N_FOLDS = 5
def train_counts(train_rows):
    """Counts over the training text, and the same with each fold left out."""
    total, folds, fold_of = Counts(), [Counts() for _ in range(N_FOLDS)], {}
    for i, row in enumerate(train_rows):
        k = min(N_FOLDS - 1, i * N_FOLDS // len(train_rows))
        fold_of[row["row_id"]] = k
        for t in (row["toks"], tok(row["before"])):
            total.add(t); folds[k].add(t)
    return total, [Minus(total, f) for f in folds], fold_of

def shape(t):
    if t.isdigit(): return "D"
    if re.fullmatch(r"[ivxlc]+", t): return "r"
    if t.isalpha(): return "A" if t[0].isupper() else ("a1" if len(t) == 1 else "a")
    if t.isalnum(): return "x"
    return t

def char_bigrams(s):
    s = s.lower()
    return {s[i:i+2] for i in range(len(s)-1)} or {s}

LM_KEYS = ("lm_in", "tri_in", "tri_in2", "lm_out", "tri_out", "lm_out0")

def pair_features(row, si, f, c):
    """Counts `c` from train, plus the same count features from the row's own text."""
    d = _features(row, si, f, c)
    if "local" not in row:
        row["local"] = Counts([row["toks"]])
    e = _features(row, si, f, row["local"])
    for k in LM_KEYS:
        if k in e: d[k + "_loc"] = e[k]
    return d

def _features(row, si, f, c):
    s, g = row["toks"], row["slots"][si]
    L1 = s[g-1] if g > 0 else "<s>";  L2 = s[g-2] if g > 1 else "<s>"
    R1 = s[g] if g < len(s) else "</s>"; R2 = s[g+1] if g+1 < len(s) else "</s>"
    F1, Fl, lb = f[0], f[-1], min(len(f), 6)
    d = {f"L1F1={L1}|{F1}": 1, f"L2L1F1={L2}|{L1}|{F1}": 1, f"FlR1={Fl}|{R1}": 1,
         f"F1R1={F1}|{R1}": 1, f"len={lb}": 1, f"lenR1={lb}|{R1}": 1, f"lenL1={lb}|{L1}": 1,
         f"sh={shape(F1)}|{shape(R1)}": 1, f"shL={shape(L1)}|{shape(F1)}": 1,
         f"slot={L1}|{R1}": 1, f"slotsh={shape(L1)}|{shape(R1)}": 1,
         f"F1={F1}|len{lb}": 1, f"eqF1R1={F1 == R1}": 1, f"eqFlL1={Fl == L1}": 1}
    if len(f) > 1:
        d[f"F2R={f[1]}|{R2}"] = 1
        d["tri_in2"] = math.log1p(c.tri.get((L1, F1, f[1]), 0))
    # does the fragment read on from what precedes the mark?
    d["lm_in"] = math.log1p(c.big.get((L1, F1), 0)) - 0.5 * math.log1p(c.uni.get(L1, 0))
    d["tri_in"] = math.log1p(c.tri.get((L2, L1, F1), 0))
    # does the text read on from the fragment, allowing for replaced wording?
    out2 = out3 = 0.0
    for k in range(10):
        if k < 8 and g + k < len(s):
            out2 = max(out2, math.log1p(c.big.get((Fl, s[g+k]), 0)) - 0.3 * k)
        if g + k + 1 < len(s):
            out3 = max(out3, math.log1p(c.tri.get((Fl, s[g+k], s[g+k+1]), 0)) - 0.2 * k)
    d["lm_out"], d["tri_out"] = out2, out3
    d["lm_out0"] = math.log1p(c.big.get((Fl, R1), 0))
    # a replacement tends to look like the wording it displaced
    nxt = s[g:g + max(1, len(f))]
    if nxt:
        a, b = char_bigrams(" ".join(f)), char_bigrams(" ".join(nxt))
        j = len(a & b) / max(1, len(a | b))
        d["jac"] = j; d[f"jacb={int(j*5)}|{min(len(f), 4)}"] = 1
        d[f"lowereq={F1.lower() == nxt[0].lower()}"] = 1
        d[f"pfx={F1[:3].lower() == nxt[0][:3].lower()}|{min(len(F1), 4)}"] = 1
    # an amendment's edits sit together
    sl = row["slots"]
    nd = min([abs(g - sl[j]) for j in (si-1, si+1) if 0 <= j < len(sl)] or [999])
    nb = min(int(math.log2(nd + 1)), 7)
    d[f"nd={nb}"] = 1; d[f"nd_len={nb}|{min(len(f), 4)}"] = 1
    d[f"relpos={int(10 * g / max(1, len(s)))}"] = 1
    return d

def assign(row, P):
    ri, ci = linear_sum_assignment(-P)
    return render(row["pieces"], {int(k): [row["frags"][i]] for i, k in zip(ri, ci)})

def make_stage2(c):
    def predict(row):
        fl = [tok(f) for f in row["frags"]]
        P = np.zeros((len(fl), len(row["slots"])))
        for i, f in enumerate(fl):
            for k in range(len(row["slots"])):
                d = pair_features(row, k, f, c)
                P[i, k] = d["lm_in"] + d["lm_out"] + d["lm_in_loc"] + d["lm_out_loc"]
        return assign(row, P)
    return predict

# validation: counts from the fit rows only
c_val, oof_val, fold_val = train_counts(FIT)
v2 = evaluate(make_stage2(c_val))
print("bigram heuristic (val):", round(v2, 4))

# test: counts from all of train; test.csv is never used as a corpus
c_all, oof_all, fold_all = train_counts(TR)
stage2 = make_stage2(c_all)
pd.DataFrame({"row_id": [r["row_id"] for r in TE], "text_before": [stage2(r) for r in TE]}) \
  .to_csv(os.path.join(OUT, "submission.csv"), index=False)
print("wrote submission.csv")

bigram heuristic (val): 0.2615


wrote submission.csv


## Submission 3 — learned pair scorer + assignment

`train_slots.csv` says which mark every training fragment came out of, so every
(mark, fragment) pair of a training row is a labelled example: one true mark per
fragment, the rest false. A logistic regression over hashed pair features learns what
the counts cannot — which kinds of fragment go with which kinds of place, what a
replacement looks like next to the wording it displaced, how close real marks sit.

One care: a training row's own prior wording contains its fragments in place, so counts
that include it make every true pair look better than it will at test time. Training
features therefore use counts with the row's own fold of `train.csv` left out.

In [6]:
from sklearn.feature_extraction import FeatureHasher
from sklearn.linear_model import LogisticRegression

def make_stage3(train_rows, c, oof, fold_of, C=1.0):
    hasher = FeatureHasher(n_features=2**20)
    X, y = [], []
    for row in train_rows:
        c_row = oof[fold_of[row["row_id"]]]
        for fi, frag in enumerate(row["frags"]):
            f = tok(frag)
            for k in range(len(row["slots"])):
                X.append(pair_features(row, k, f, c_row))
                y.append(int(labels[row["row_id"]][fi] == k))
    clf = LogisticRegression(C=C, max_iter=500, class_weight="balanced")
    clf.fit(hasher.transform(X), y)
    print(f"  trained on {len(y):,} pairs, {sum(y):,} true")
    def predict(row):
        fl = [tok(f) for f in row["frags"]]
        P = np.array([clf.decision_function(hasher.transform(
            [pair_features(row, k, f, c) for k in range(len(row["slots"]))])) for f in fl])
        return assign(row, P)
    return predict

v3 = evaluate(make_stage3(FIT, c_val, oof_val, fold_val))
print("marks removed (val):    0.0")
print("bigram heuristic (val):", round(v2, 4))
print("pair scorer (val):     ", round(v3, 4))

  trained on 866,220 pairs, 3,992 true


marks removed (val):    0.0
bigram heuristic (val): 0.2615
pair scorer (val):      0.4766


In [7]:
# Refit on ALL of train, then predict the test set.
stage3 = make_stage3(TR, c_all, oof_all, fold_all)
preds = [stage3(r) for r in TE]
assert all(MARK not in p for p in preds)
pd.DataFrame({"row_id": [r["row_id"] for r in TE], "text_before": preds}) \
  .to_csv(os.path.join(OUT, "submission.csv"), index=False)
print(f"wrote submission.csv — {len(preds)} rows")

  trained on 936,896 pairs, 4,631 true


wrote submission.csv — 353 rows


## What to try next

- **A real language model of this register.** The counts are bigrams and trigrams over
  the training text. A small transformer trained from scratch on the same text,
  scoring each candidate placement by how the whole sentence reads, is the obvious next
  step — single-token fragments are where word counts stop discriminating.
- **Score the row jointly.** The assignment treats pairs as independent. Real marks sit
  together, paragraph designations run in sequence, and a fragment's neighbours
  constrain it; a model that scores a whole placement would use that.
- **Work out what was replaced.** Where wording was substituted, the replacement is
  still in the text after the mark. Guessing its extent gives the fragment its true
  right-hand context instead of a new one.
- **Hedge deliberately.** The metric is an F1, so placing a fragment at its two best
  marks can beat committing to one when the scorer is unsure — and costs precision when
  it is not.